In [0]:
# COMMAND ----------
# STEP 1: INITIALIZE LIBRARIES & DEFINE ENVIRONMENTAL CONFIGURATIONS
# COMMAND ----------
from pyspark.sql.functions import current_timestamp

# Clean, version-controlled paths to your Unity Catalog Volume and Schema
VOLUME_PATH = "/Volumes/dev_catalog/customer_analytics/landing"
TARGET_SCHEMA = "dev_catalog.customer_analytics"


In [0]:
from pyspark.sql.functions import col, current_timestamp

# Define paths
landing_path = f"{VOLUME_PATH}/Customers/"
archive_dir = f"{VOLUME_PATH}/Customers/Archive/"

# 1. Read top-level files only (ignoring the Archive subfolder)
raw_customers_df = (
    spark.read
    .option("recursiveFileLookup", "false")
    .format("parquet")
    .load(landing_path)
    .withColumn("bronze_ingestion_time", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
)

# Check if any files were loaded before proceeding
if not raw_customers_df.isEmpty():
    
    # 2. Extract the exact list of file paths read in this batch
    files_to_move = [
        row.source_file_name 
        for row in raw_customers_df.select("source_file_name").distinct().collect()
    ]

    # 3. Append data to Bronze Delta Table
    (
        raw_customers_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{TARGET_SCHEMA}.bronze_customers")
    )

    print(f"✅ Customers data appended successfully. Total row count: {spark.table(f'{TARGET_SCHEMA}.bronze_customers').count()}")

    # 4. Ensure Archive directory exists
    dbutils.fs.mkdirs(archive_dir)

    # 5. Move each processed file into Archive
    for file_path in files_to_move:
        file_name = file_path.split("/")[-1]
        dbutils.fs.mv(file_path, f"{archive_dir}{file_name}")

    print(f"📦 Successfully archived {len(files_to_move)} file(s) to {archive_dir}")

else:
    print("ℹ️ No new files found to process.")

In [0]:
# COMMAND ----------
# STEP 3: INGEST ORDERS VIA BATCH LOAD & ARCHIVE PROCESSED FILES
# COMMAND ----------
from pyspark.sql.functions import col, current_timestamp

# Define landing and archive paths
orders_landing_path = f"{VOLUME_PATH}/Orders/"
orders_archive_dir = f"{VOLUME_PATH}/Orders/Archive/"

# 1. Read top-level files only (ignoring the Archive subfolder)
raw_orders_df = (
    spark.read
    .option("recursiveFileLookup", "false")
    .format("parquet")
    .load(orders_landing_path)
    .withColumn("bronze_ingestion_time", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
)

# Process only if files exist in the landing directory
if not raw_orders_df.isEmpty():
    
    # 2. Extract unique source file paths BEFORE writing
    files_to_move = [
        row.source_file_name 
        for row in raw_orders_df.select("source_file_name").distinct().collect()
    ]

    # 3. Append safely to the historical ledger table
    (
        raw_orders_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{TARGET_SCHEMA}.bronze_orders")
    )

    print(f"✅ Orders data appended successfully. Total historical row count: {spark.table(f'{TARGET_SCHEMA}.bronze_orders').count()}")

    # 4. Ensure Orders Archive directory exists
    dbutils.fs.mkdirs(orders_archive_dir)

    # 5. Move ingested files to Archive
    for file_path in files_to_move:
        file_name = file_path.split("/")[-1]
        dbutils.fs.mv(file_path, f"{orders_archive_dir}{file_name}")

    print(f"📦 Successfully archived {len(files_to_move)} order file(s) to {orders_archive_dir}")

else:
    print("ℹ️ No new orders files found to process.")